# 02 — Tận dụng phần còn thiếu từ AI Guru (Colab CPU)

**CPU → Run all**, một phiên trên cùng output folder. Không đổi cấu hình.
Cell 3 đưa text đã khôi phục vào dữ liệu chuẩn trước. Cell 4 tiếp tục cứu
raw đã crawl; có thể ngắt sau checkpoint và Run all tiếp trong phiên mới.

Không mở lại corpus/vector/BM25 đã xử lý, không đổi candidate luồng submit.
Dùng cùng tokenizer/chunk policy và producer code đã khóa. Không gọi model
embedding, không gọi lại website. BGE ở đây chỉ là tokenizer CPU.

Raw SUCCESS không chứng minh có bài viết: hash/length phải khớp, parser và
gate nội dung phải qua. Trang challenge, lỗi, retained LOW/QUARANTINE được
ghi riêng để review/crawl tiếp. Không hứa phục hồi được toàn bộ raw.

Mỗi source metadata được lập hàng đợi một lần. TAR raw cần một lượt đọc
header để tìm đúng member; cursor được lưu trên Drive mỗi 1.024 header hoặc
32 text mới. Bản raw cùng hash + URL + trạng thái review không parse lại.
Source/cursor đã commit được dùng lại ở runtime khác. Đoạn đang ghi chưa
commit có thể làm lại khi bị ngắt. Không quét lại cả TAR chỉ để xác minh.

Kết quả riêng: `VietMedBridge/data_staging/btc-maximize-71be0e12-v1/`.
`delta_candidates.json`: candidate + model inputs của dữ liệu mới.
`remaining_for_crawl.csv`: mọi ID còn thiếu, kèm nguyên nhân.
`raw_recovery_outcomes.csv`: hash/URL đã thử và kết quả thật.
`checkpoint.json` + SQL snapshot: tiếp tục qua phiên/tài khoản khác.

Dữ liệu tạm chỉ ở máy Colab, không tải về PC. Không copy nguyên TAR/raw ra
ổ Colab hoặc Drive; chỉ đọc raw cần thử trong bộ nhớ, lưu text/metadata và
chunks đã xử lý. Dữ liệu cũ và luồng submit đang chạy được giữ nguyên.

In [ ]:
#@title 1. Colab CPU — kết nối Drive, code/parser/chunk policy cố định
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys
print('MAXIMIZE: CPU only; preserve paid corpus and active submission.',flush=True)
if not Path('/var/colab/hostname').is_file():
    raise RuntimeError('Chạy notebook này trên Colab CPU.')
from google.colab import drive,auth
drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/VietMedBridge')
relative='data_staging/btc-all-ai-guru-71be0e12-v1/collection_report.json'
if not (PROJECT/relative).is_file():
    PROJECT=Path('/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge')
if not (PROJECT/relative).is_file():
    raise FileNotFoundError('Chưa thấy báo cáo collection đã hoàn tất trong VietMedBridge.')
PROJECT=PROJECT.resolve()
WORK=Path('/content/vmb_maximize_work')
WORK.mkdir(parents=True,exist_ok=True)
CHECKOUT=Path('/content/VietMedBridge-maximize')
REPO='https://github.com/Platypus27-coder/VietMedBridge.git'
COMMIT='bbf09ec01d542a5b06a4c2beec245571abeb3515'
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git','clone','--no-checkout',REPO,str(CHECKOUT)],check=True)
if subprocess.check_output(['git','-C',str(CHECKOUT),'remote','get-url','origin'],text=True).strip()!=REPO:
    raise ValueError('Unexpected checkout repository.')
subprocess.run(['git','-C',str(CHECKOUT),'fetch','--depth','1','origin',COMMIT],check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(CHECKOUT),'numpy'],check=True)
runtime_lock=PROJECT/'data_staging/btc-maximize-71be0e12-v1/runtime_lock.json'
if runtime_lock.is_file():
    pinned=json.loads(runtime_lock.read_text())
    subprocess.run([sys.executable,'-m','pip','install',*[k+'=='+v for k,v in pinned.items()
        if k!='vietmedbridge']],check=True)
sys.path[:0]=[str(WORK),str(CHECKOUT/'src')]
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json,runtime_versions
if not runtime_lock.is_file(): atomic_json(runtime_lock,runtime_versions())
os.environ['HF_HOME']='/content/hf_cache_maximize'
os.environ['TOKENIZERS_PARALLELISM']='false'
auth.authenticate_user()

In [ ]:
#@title 2. Chỉ nhận checkpoint đã chốt; không đọc corpus/vector cũ
COLLECT_SOURCE='"""Collect all available BTC URL content from every AI Guru archive version.\n\nRead metadata members through Drive Range, never unpack raw TARs. Original\nsnapshot/hash provenance is preserved; current BTC membership is a separate join.\nThis collects extraction candidates, not silently promoted chunks or embeddings.\n"""\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom email.utils import parsedate_to_datetime\nfrom pathlib import Path, PurePosixPath\nimport gzip\nimport hashlib\nimport json\nimport re\nimport shutil\nimport sqlite3\nimport tarfile\nimport tempfile\nimport time\n\nimport duckdb\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport requests\nfrom urllib3.exceptions import ProtocolError, ReadTimeoutError\nfrom vietmedbridge.artifacts import atomic_json, digest_json, publish_file, read_json, sha256_file, verify_file, utc_now\nfrom vietmedbridge.content_embeddings import checked, seal\nfrom vietmedbridge.external_import import _hold, MEMBERS\nfrom vietmedbridge.isolated_intake import Window, DriveRange, DriveSource\n\nFOLDER = \'1P2ke-yIDwj443R_GMegaiH9RACf-Gzth\'\nOFFICIAL_SHA = \'87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7\'\nCAMPAIGN = \'btc-all-ai-guru-71be0e12-v1\'\nRAW_PATH_POLICY = \'native-data-raw-and-portable-raw-v2\'\n\n\ndef raw_record_problem(row, leaf_format, descriptor):\n    """Validate provenance without confusing a path/schema problem with HTTP failure."""\n    if row[\'status\'] != \'SUCCESS_RAW\':\n        return row[\'status\'] or \'MISSING_CRAWL_RESULT\'\n    if not isinstance(row[\'raw_sha256\'],str) or not re.fullmatch(\'[0-9a-f]{64}\',row[\'raw_sha256\']):\n        return \'RAW_RECORD_INVALID_SHA256\'\n    path=row[\'raw_path\']\n    prefixes=(\'raw/\',\'data/raw/\') if leaf_format==\'native\' else (\'raw/\',)\n    if (not isinstance(path,str) or not path.startswith(prefixes) or path.endswith(\'/\') or \'\\\\\' in path or \'\\x00\' in path\n            or PurePosixPath(path).is_absolute() or \'..\' in path.split(\'/\')):\n        return \'RAW_RECORD_INVALID_LOCATOR\'\n    if not row[\'crawl_snapshot\']:\n        return \'RAW_RECORD_MISSING_SNAPSHOT\'\n    if leaf_format==\'native\' and row[\'crawl_snapshot\'] != descriptor.get(\'snapshot_id\'):\n        return \'ARCHIVE_SNAPSHOT_BINDING_MISMATCH\'\n    return None\n\n\ndef repair_leaf_id(base_id):\n    return digest_json({\'original_leaf\':base_id,\'raw_path_policy\':RAW_PATH_POLICY})\n\n\nclass CollectionDriveError(RuntimeError):\n    def __init__(self, status, reasons, file_id, byte_range, *, cooldown=0, pause=False):\n        self.details = {\'http_status\':status, \'reasons\':list(reasons), \'file_id\':file_id,\n                        \'byte_range\':byte_range, \'cooldown_seconds\':cooldown, \'pause_campaign\':pause}\n        super().__init__(f\'Drive HTTP {status} | reason={",".join(reasons) or "UNKNOWN"} | \'\n            f\'file={file_id} | Range={byte_range} | cooldown={cooldown:g}s | checkpoint retained\')\n\n\ndef drive_reasons(response):\n    # Error bodies only, bounded; never print tokens, headers or private bodies.\n    try:\n        body=json.loads(response.raw.read(65536))\n        error=body.get(\'error\',{}) if isinstance(body,dict) else {}\n        if not isinstance(error,dict):\n            return ()\n        entries=[*(error.get(\'errors\') or []),*(error.get(\'details\') or [])]\n        return tuple(sorted({r[\'reason\'] for r in entries if isinstance(r,dict)\n            and isinstance(r.get(\'reason\'),str) and re.fullmatch(r\'[A-Za-z][A-Za-z0-9_.-]{0,99}\',r[\'reason\'])}))\n    except (ValueError,TypeError,AttributeError):\n        return ()\n\n\ndef drive_delay(response, fallback):\n    value=response.headers.get(\'Retry-After\',\'\').strip()\n    if value.isdigit():\n        return max(fallback,int(value))\n    if value:\n        try:\n            until=parsedate_to_datetime(value)\n            if until.tzinfo is None:\n                until=until.replace(tzinfo=timezone.utc)\n            return max(fallback,(until-datetime.now(timezone.utc)).total_seconds())\n        except (ValueError,TypeError,OverflowError):\n            pass\n    return fallback\n\n\nclass CollectionDriveRange(DriveRange):\n    """Retry the identical Range only for explicit transient provider failures."""\n    def _get_block(self,start):\n        limit=getattr(self,\'range_end\',self.size)\n        if not 0<=start<limit<=self.size:\n            raise ValueError(\'Drive range exceeds the selected metadata member.\')\n        end=min(limit,start+self.block_size)-1\n        requested=f\'bytes={start}-{end}\'\n        # Google may send legacy error.errors and google.rpc.ErrorInfo together.\n        # Both spellings must be recognized; permission/quota codes stay denied.\n        rate_reasons={\'rateLimitExceeded\',\'userRateLimitExceeded\',\n                      \'RATE_LIMIT_EXCEEDED\',\'USER_RATE_LIMIT_EXCEEDED\'}\n        for attempt in range(4):\n            delay=2**attempt\n            try:\n                with self.session.get(self.url,params={\'alt\':\'media\',\'supportsAllDrives\':\'true\'},\n                        headers={\'Range\':requested,\'Accept-Encoding\':\'identity\'},stream=True,timeout=(15,90)) as response:\n                    status=response.status_code\n                    if status>=400:\n                        reasons=drive_reasons(response)\n                        retryable=status in (429,500,502,503,504) or (status==403 and bool(reasons)\n                            and set(reasons)<=rate_reasons)\n                        delay=drive_delay(response,delay)\n                        print(\'DRIVE_RANGE_ERROR:\',status,\'| reason=\'+(\',\'.join(reasons) or \'UNKNOWN\'),\n                              \'|\',self.file_id,\'|\',requested,flush=True)\n                        if not retryable or attempt==3 or delay>60:\n                            pause=status==429 or (bool(reasons) and set(reasons)<=rate_reasons|{\'dailyLimitExceeded\'})\n                            raise CollectionDriveError(status,reasons,self.file_id,requested,cooldown=delay,pause=pause)\n                    else:\n                        expected=f\'bytes {start}-{end}/{self.size}\'\n                        if status!=206 or response.headers.get(\'Content-Range\')!=expected:\n                            raise ValueError(\'Drive did not honor the bounded Range request; no full TAR consumed.\')\n                        payload=response.raw.read(end-start+2)\n                        if len(payload)!=end-start+1:\n                            raise EOFError(\'Incomplete Drive range; unpublished block retained for retry.\')\n                        self.block_start,self.block=start,payload\n                        return\n            except (requests.ConnectionError,requests.Timeout,requests.exceptions.ChunkedEncodingError,\n                    ProtocolError,ReadTimeoutError,EOFError):\n                if attempt==3:\n                    raise\n            print(f\'DRIVE_RANGE_RETRY {attempt+1}/3 | same {requested} | wait {delay:g}s\',flush=True)\n            time.sleep(delay)\n\n\nclass CollectionDriveSource(DriveSource):\n    def open(self,file_id,metadata):\n        return CollectionDriveRange(self.session,file_id,metadata)\n\n\ndef fetch_url(value):\n    return value.strip().split(\'#\', 1)[0]\n\n\ndef inventory(session, folder=FOLDER):\n    result, visited, queue = [], set(), [folder]\n    while queue:\n        current = queue.pop(0)\n        if current in visited:\n            continue\n        visited.add(current)\n        token = None\n        while True:\n            params = {\'q\': f"\'{current}\' in parents and trashed=false", \'pageSize\': 1000,\n                      \'fields\': \'nextPageToken,files(id,name,mimeType,size,modifiedTime,md5Checksum,shortcutDetails)\',\n                      \'supportsAllDrives\': \'true\', \'includeItemsFromAllDrives\': \'true\'}\n            if token:\n                params[\'pageToken\'] = token\n            response = session.get(\'https://www.googleapis.com/drive/v3/files\', params=params, timeout=90)\n            response.raise_for_status()\n            body = response.json()\n            for item in body.get(\'files\', []):\n                if item[\'mimeType\'] == \'application/vnd.google-apps.shortcut\':\n                    target = item[\'shortcutDetails\']\n                    response = session.get(\'https://www.googleapis.com/drive/v3/files/\'+target[\'targetId\'], params={\n                        \'fields\': \'id,name,mimeType,size,modifiedTime,md5Checksum\', \'supportsAllDrives\': \'true\'}, timeout=90)\n                    response.raise_for_status()\n                    item = response.json()\n                if item[\'mimeType\'] == \'application/vnd.google-apps.folder\':\n                    queue.append(item[\'id\'])\n                elif item[\'name\'].lower().endswith(\'.tar\'):\n                    if not item.get(\'size\') or not item.get(\'md5Checksum\'):\n                        raise ValueError(\'Archive has no immutable byte identity.\')\n                    result.append(item)\n            token = body.get(\'nextPageToken\')\n            if not token:\n                break\n    return sorted({r[\'id\']: r for r in result}.values(), key=lambda r: (r.get(\'modifiedTime\', \'\'), r[\'id\']), reverse=True)\n\n\ndef identity(remote):\n    return {k: remote[k] for k in (\'id\', \'name\', \'size\', \'modifiedTime\', \'md5Checksum\')}\n\n\n@contextmanager\ndef metadata_leaves(reader, remote, retained, work):\n    """Read native leaf metadata, or enumerate every child TAR of a container.\n\n    A native source\'s four manifests are its authoritative URL inventory. Its\n    raw members are payloads, not additional sources, and must not be enumerated\n    after the manifests are complete. Containers have no native manifests; walk\n    their complete member list so later resume TARs are still discovered.\n    """\n    with reader.open(remote[\'id\'], identity(remote)) as stream:\n        if hasattr(stream, \'block_size\'):\n            stream.block_size = 65536\n        leaves = []\n        def scan(start, size, name, depth=0):\n            if depth > 3:\n                raise ValueError(\'Nested archive depth exceeds the bounded scanner.\')\n            found, nested = {}, []\n            headers, logged = 0, time.monotonic()\n            with tarfile.open(fileobj=Window(stream, start, size), mode=\'r:\') as archive:\n                for member in archive:\n                    headers += 1\n                    if time.monotonic()-logged >= 15:\n                        print(\'ARCHIVE_HEADER_PROGRESS:\',name,\'|\',headers,\'headers | metadata\',len(found),\'/4\',flush=True)\n                        logged = time.monotonic()\n                    if not member.isfile():\n                        continue\n                    label = (\'dataset\' if member.name == MEMBERS[\'dataset\'] else\n                             \'crawl\' if member.name == MEMBERS[\'crawl\'] else\n                             \'extracted\' if member.name == MEMBERS[\'extracted\'] else\n                             \'frontier\' if re.fullmatch(r\'data/crawl_shards/shard_[0-9]+[.]parquet\', member.name) else None)\n                    if label:\n                        if nested:\n                            raise ValueError(\'Mixed native/container layout; do not choose an arbitrary source.\')\n                        if label in found or not 0 < member.size <= 2*1024**3:\n                            raise ValueError(\'Duplicate/oversize metadata member; no arbitrary version selected.\')\n                        found[label] = {\'path\': member.name, \'offset\': start+member.offset_data,\n                                        \'bytes\': member.size, \'sha256\': member.pax_headers.get(\'vibiomir.sha256\')}\n                        if len(found) == 4:\n                            print(\'NATIVE_METADATA_READY:\',name,\'|\',headers,\n                                  \'headers | 4/4 manifests; raw member listing skipped\',flush=True)\n                            break\n                    elif member.name.endswith(\'.tar\') and not member.name.startswith(\'raw/\'):\n                        if found:\n                            raise ValueError(\'Mixed native/container layout; source requires inspection.\')\n                        nested.append((start+member.offset_data, member.size, name+\'!\'+member.name))\n            if found:\n                if set(found) != {\'dataset\',\'crawl\',\'extracted\',\'frontier\'}:\n                    raise ValueError(\'Incomplete leaf metadata; preserve this archive for inspection.\')\n                leaves.append({\'name\': name, \'format\': \'native\', \'members\': found})\n            for entry in nested:\n                scan(*entry, depth+1)\n        if remote[\'id\'] == retained[\'drive_file_id\']:\n            if int(remote[\'size\']) != retained[\'archive_bytes\']:\n                raise ValueError(\'Pinned retained package changed; do not guess member offsets.\')\n            members = retained[\'members\']\n            for item in members.values():\n                stream.seek(item[\'offset\']-512)\n                header = tarfile.TarInfo.frombuf(stream.read(512), \'utf-8\', \'surrogateescape\')\n                if header.name != item[\'path\'] or header.size != item[\'bytes\'] or not header.isfile():\n                    raise ValueError(\'Retained member header changed.\')\n            leaves.append({\'name\': remote[\'name\'], \'format\': \'retained\', \'members\': members})\n        else:\n            scan(0, int(remote[\'size\']), remote[\'name\'])\n        if not leaves:\n            raise ValueError(\'No supported metadata leaf discovered in this archive.\')\n        yield stream, leaves\n\n\ndef stage_member(stream, item, target):\n    if hasattr(stream, \'block_size\'):\n        stream.block_size = 8*1024**2\n        stream.block_start, stream.block = -1, b\'\'\n    stream.seek(item[\'offset\'])\n    digest, remaining = hashlib.sha256(), item[\'bytes\']\n    logged = time.monotonic()\n    with Path(target).open(\'wb\') as out:\n        while remaining:\n            payload = stream.read(min(8*1024**2, remaining))\n            if not payload:\n                raise EOFError(\'Truncated metadata member.\')\n            out.write(payload)\n            digest.update(payload)\n            remaining -= len(payload)\n            if time.monotonic()-logged>=15 or remaining==0:\n                print(\'METADATA_TRANSFER\',item[\'path\'],\'|\',round((item[\'bytes\']-remaining)/1024**2,1),\n                      \'/\',round(item[\'bytes\']/1024**2,1),\'MiB\',flush=True)\n                logged=time.monotonic()\n    if item.get(\'sha256\') and item[\'sha256\'] != digest.hexdigest():\n        raise ValueError(\'Source metadata checksum differs from the TAR receipt.\')\n    return digest.hexdigest()\n\n\ndef cached_member(stream,item,target,cache_dir):\n    """Keep completed metadata members on Drive even if a later request fails.\n\n    An in-flight Range retries in place. An incomplete member without a receipt\n    may restart after a hard failure; complete members never reopen the TAR.\n    """\n    cache_dir=Path(cache_dir)\n    receipt=cache_dir/\'receipt.json\'\n    payload=cache_dir/\'metadata.bin\'\n    binding=digest_json(item)\n    if receipt.is_file():\n        saved=checked(read_json(receipt))\n        if saved[\'binding\']!=binding or saved[\'bytes\']!=item[\'bytes\']:\n            raise ValueError(\'Metadata member cache belongs to another source version.\')\n        shutil.copyfile(payload,target)\n        verify_file(target,saved[\'sha256\'])\n        if Path(target).stat().st_size!=item[\'bytes\']:\n            raise ValueError(\'Cached metadata member length differs from its TAR header.\')\n        if item.get(\'sha256\') and saved[\'sha256\']!=item[\'sha256\']:\n            raise ValueError(\'Cached metadata differs from source TAR checksum.\')\n        print(\'REUSED_METADATA_FROM_DRIVE:\',item[\'path\'],\'| no TAR payload read\',flush=True)\n        return saved[\'sha256\']\n    checksum=stage_member(stream,item,target)\n    if publish_file(target,payload)!=checksum:\n        raise ValueError(\'Metadata cache publication checksum mismatch.\')\n    atomic_json(receipt,seal({\'binding\':binding,\'bytes\':item[\'bytes\'],\'sha256\':checksum}))\n    print(\'METADATA_CHECKPOINT_SAVED_ON_DRIVE:\',item[\'path\'],flush=True)\n    return checksum\n\n\nclass CollectAll:\n    def __init__(self, project, work, *, official_sha=OFFICIAL_SHA):\n        self.project, self.work = Path(project).resolve(), Path(work).resolve()\n        self.root = self.project/\'data_staging\'/CAMPAIGN\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.work.mkdir(parents=True, exist_ok=True)\n        self.binding = {\'schema\': \'whole-folder-content-collection-v1\', \'official_sha256\': official_sha,\n                        \'folder_id\': FOLDER, \'normalization\': \'trim-and-remove-fragment-only\'}\n        self.database = self.work/\'collection.sqlite\'\n        self.pointer = self.root/\'checkpoint.json\'\n        if not self.database.exists() and self.pointer.exists():\n            saved = checked(read_json(self.pointer))\n            if saved[\'binding\'] != self.binding:\n                raise ValueError(\'Collection BTC snapshot changed.\')\n            payload = self.root/saved[\'path\']\n            verify_file(payload, saved[\'sha256\'])\n            print(\'RESTORE_COLLECTION: one SQL checkpoint; no old archives reopened\', flush=True)\n            with gzip.open(payload, \'rb\') as source, self.database.open(\'wb\') as out:\n                shutil.copyfileobj(source, out, length=1024**2)\n            verify_file(self.database, saved[\'sqlite_sha256\'])\n        self.con = sqlite3.connect(self.database)\n        self.con.execute(\'PRAGMA cache_size=-32768\')\n        self.con.execute(\'PRAGMA synchronous=FULL\')\n        self.con.execute(\'CREATE TABLE IF NOT EXISTS contract(value TEXT)\')\n        row = self.con.execute(\'SELECT value FROM contract\').fetchone()\n        if row and row[0] != digest_json(self.binding):\n            raise ValueError(\'Retained local collection belongs to another BTC snapshot.\')\n        if not row:\n            self.con.execute(\'INSERT INTO contract VALUES (?)\', [digest_json(self.binding)])\n        self.con.executescript(\'\'\'\n            CREATE TABLE IF NOT EXISTS official(id INTEGER PRIMARY KEY,url TEXT NOT NULL,fetch TEXT NOT NULL);\n            CREATE INDEX IF NOT EXISTS official_fetch ON official(fetch);\n            CREATE TABLE IF NOT EXISTS content(fetch TEXT PRIMARY KEY,seen INTEGER DEFAULT 0,raw INTEGER DEFAULT 0,\n                existing TEXT,candidate TEXT,reason TEXT);\n            CREATE TABLE IF NOT EXISTS completed(id TEXT PRIMARY KEY,kind TEXT,receipt TEXT);\n        \'\'\')\n        columns={r[1] for r in self.con.execute(\'PRAGMA table_info(content)\')}\n        if \'raw_status\' not in columns:\n            self.con.execute(\'ALTER TABLE content ADD COLUMN raw_status INTEGER DEFAULT 0\')\n            self.con.execute(\'UPDATE content SET raw_status=raw\')\n        self.con.commit()\n\n    def saved(self, identifier):\n        row = self.con.execute(\'SELECT receipt FROM completed WHERE id=?\', [identifier]).fetchone()\n        return json.loads(row[0]) if row else None\n\n    def commit(self, identifier, kind, receipt):\n        self.con.execute(\'INSERT INTO completed VALUES (?,?,?)\', [identifier, kind, json.dumps(receipt)])\n        self.con.commit()\n        self.snapshot()\n\n    def snapshot(self):\n        copy = self.work/\'backup.sqlite\'\n        destination = sqlite3.connect(copy)\n        try:\n            self.con.backup(destination)\n        finally:\n            destination.close()\n        payload = self.work/\'backup.sqlite.gz\'\n        with copy.open(\'rb\') as source, gzip.open(payload, \'wb\', compresslevel=1) as out:\n            shutil.copyfileobj(source, out, length=1024**2)\n        checksum = sha256_file(payload)\n        target = self.root/(\'sql-\'+checksum+\'.gz\')\n        publish_file(payload, target)\n        atomic_json(self.pointer, seal({\'binding\': self.binding, \'path\': target.name, \'sha256\': checksum,\n            \'sqlite_sha256\': sha256_file(copy), \'updated_at\': utc_now()}))\n        copy.unlink()\n        payload.unlink()\n        print(\'COLLECTION_SQL_CHECKPOINT_SAVED_ON_DRIVE\', flush=True)\n\n    def official(self, path):\n        if self.saved(\'official\'):\n            return\n        verify_file(path, self.binding[\'official_sha256\'])\n        try:\n            for batch in pq.ParquetFile(path).iter_batches(batch_size=8192, columns=[\'id\',\'url\']):\n                rows = [(r[\'id\'], r[\'url\'], fetch_url(r[\'url\'])) for r in batch.to_pylist()]\n                self.con.executemany(\'INSERT INTO official VALUES (?,?,?)\', rows)\n                self.con.executemany(\'INSERT OR IGNORE INTO content(fetch) VALUES (?)\', [(r[2],) for r in rows])\n            self.commit(\'official\', \'reference\', {\'sha256\': self.binding[\'official_sha256\']})\n        except BaseException:\n            self.con.rollback()\n            raise\n\n    def existing(self, path, checksum, label):\n        identifier = \'existing:\'+checksum\n        if self.saved(identifier):\n            print(\'SKIP_EXISTING_ALIAS_MAPPING:\', label, flush=True)\n            return\n        verify_file(path, checksum)\n        try:\n            for batch in pq.ParquetFile(path).iter_batches(batch_size=8192, columns=[\'doc_id\',\'url\']):\n                rows = [(json.dumps({\'mapping\': label, \'original_doc_id\': r[\'doc_id\'], \'original_url\': r[\'url\']}), fetch_url(r[\'url\']))\n                        for r in batch.to_pylist()]\n                self.con.executemany(\'UPDATE content SET existing=coalesce(existing,?) WHERE fetch=?\', rows)\n            self.commit(identifier, \'existing\', {\'mapping\': label, \'sha256\': checksum})\n        except BaseException:\n            self.con.rollback()\n            raise\n\n    def leaf(self, stream, remote, leaf, *, version_check=None, repair=False, metadata_paths=None):\n        base_id = digest_json({\'remote\': identity(remote), \'leaf\': leaf})\n        identifier = repair_leaf_id(base_id) if repair else base_id\n        if self.saved(identifier):\n            print(\'SKIP_COMPLETED_ARCHIVE_LEAF:\', leaf[\'name\'], \'| receipt only\', flush=True)\n            return\n        print(\'COLLECT_ARCHIVE_LEAF:\', leaf[\'name\'], \'| metadata only, no raw TAR copy\', flush=True)\n        with tempfile.TemporaryDirectory(prefix=\'metadata-\', dir=self.work) as temporary:\n            local = Path(temporary)\n            staged_checksums={}\n            for kind, item in leaf[\'members\'].items():\n                if leaf[\'format\'] == \'retained\' and kind == \'mapping\':\n                    continue\n                print(\'READ_METADATA\', kind, \'|\', round(item[\'bytes\']/1024**2, 1), \'MiB\', flush=True)\n                destination=local/(kind+\'.\'+(\'json\' if kind == \'dataset\' else \'parquet\'))\n                if metadata_paths is not None:\n                    reference,checksum=metadata_paths[kind]\n                    shutil.copyfile(reference,destination)\n                    verify_file(destination,checksum)\n                    if destination.stat().st_size!=item[\'bytes\']:\n                        raise ValueError(\'Saved metadata length differs from its source receipt.\')\n                    staged_checksums[kind]=checksum\n                    print(\'REPAIR_METADATA_FROM_DRIVE:\',kind,\'| no TAR access\',flush=True)\n                else:\n                    staged_checksums[kind]=cached_member(stream,item,destination,\n                        self.root/\'metadata_cache\'/base_id/kind)\n            descriptor = read_json(local/\'dataset.json\')\n            con = duckdb.connect()\n            con.execute("SET memory_limit=\'1GB\'")\n            con.execute(\'SET threads=1\')\n            con.execute(\'SET preserve_insertion_order=false\')\n            con.execute(\'SET temp_directory=?\', [str(local/\'spill\')])\n            try:\n                if leaf[\'format\'] == \'native\':\n                    for kind in (\'frontier\',\'crawl\',\'extracted\'):\n                        con.read_parquet(str(local/(kind+\'.parquet\'))).create_view(kind)\n                        duplicates=con.execute(f\'SELECT count(*)-count(DISTINCT crawl_url_id) FROM {kind}\').fetchone()[0]\n                        if duplicates:\n                            raise ValueError(\'Duplicate crawl_url_id in source metadata; no arbitrary row selected.\')\n                    query = \'\'\'SELECT f.fetch_url,c.status,c.raw_sha256,c.raw_path,c.raw_size_bytes,c.final_url,\n                        c.content_type,c.snapshot_id AS crawl_snapshot,e.snapshot_id AS extract_snapshot,\n                        e.raw_sha256 AS extract_raw_sha,e.extract_status,e.text,e.title,e.char_count,\n                        to_json(c) AS crawl_json,to_json(e) AS extracted_json\n                        FROM frontier f LEFT JOIN crawl c USING(crawl_url_id) LEFT JOIN extracted e USING(crawl_url_id)\'\'\'\n                    if repair:\n                        # Previously failed HTTP records are already in the SQL checkpoint.\n                        # Only successful raw records can be affected by the locator bug.\n                        query += " WHERE c.status=\'SUCCESS_RAW\'"\n                else:\n                    con.read_parquet(str(local/\'raw_catalog.parquet\')).create_view(\'raw\')\n                    con.read_parquet(str(local/\'documents.parquet\')).create_view(\'docs\')\n                    for kind in (\'raw\',\'docs\'):\n                        if con.execute(f\'SELECT count(*)-count(DISTINCT crawl_url_id) FROM {kind}\').fetchone()[0]:\n                            raise ValueError(\'Duplicate retained crawl_url_id; source requires inspection.\')\n                    query = \'\'\'SELECT r.fetch_url,\'SUCCESS_RAW\' AS status,r.raw_sha256,r.raw_path,r.raw_size_bytes,r.final_url,\n                        r.content_type,r.original_snapshot_id AS crawl_snapshot,e.snapshot_id AS extract_snapshot,\n                        e.raw_sha256 AS extract_raw_sha,e.extract_status,e.text,e.title,e.char_count,\n                        r.original_crawl_record_json AS crawl_json,to_json(e) AS extracted_json,r.quality_tier AS raw_quality_tier\n                        FROM raw r LEFT JOIN docs e USING(crawl_url_id)\'\'\'\n                output = local/\'recovered.parquet\'\n                schema = pa.schema([(\'fetch_url\',pa.string()),(\'source_file_id\',pa.string()),(\'leaf\',pa.string()),\n                    (\'original_snapshot_id\',pa.string()),(\'source_text_sha256\',pa.string()),\n                    (\'crawl_json\',pa.string()),(\'extracted_json\',pa.large_string())])\n                count, matched, raw_count, held = 0, 0, 0, {}\n                logged=time.monotonic()\n                with pq.ParquetWriter(output, schema, compression=\'zstd\') as writer:\n                    for batch in con.execute(query).fetch_record_batch(128):\n                        candidates = []\n                        for row in batch.to_pylist():\n                            url = row[\'fetch_url\']\n                            state = self.con.execute(\'SELECT existing,candidate,raw FROM content WHERE fetch=?\', [url]).fetchone()\n                            if state is None:\n                                continue\n                            matched += 1\n                            original = row[\'crawl_snapshot\']\n                            raw_problem=raw_record_problem(row,leaf[\'format\'],descriptor)\n                            raw_ok=raw_problem is None\n                            raw_count += bool(raw_ok)\n                            self.con.execute(\'UPDATE content SET seen=1,raw=max(raw,?),raw_status=max(raw_status,?) WHERE fetch=?\',\n                                [int(raw_ok),int(row[\'status\']==\'SUCCESS_RAW\'),url])\n                            if state[0] is not None or state[1] is not None:\n                                continue\n                            reason = None\n                            if not raw_ok:\n                                reason = raw_problem\n                            elif row[\'extract_raw_sha\'] != row[\'raw_sha256\'] or row[\'extract_snapshot\'] != original:\n                                reason = \'MISSING_OR_UNBOUND_EXTRACTION\'\n                            elif row[\'extract_status\'] == \'EXTRACT_SUCCESS\' and len(row[\'text\'] or \'\') != row[\'char_count\']:\n                                reason = \'INVALID_SOURCE_LENGTH\'\n                            else:\n                                extra = json.loads(row[\'extracted_json\']) if row[\'extracted_json\'] else {}\n                                gate_row = {**row, \'url\':url, \'source_issue\':extra.get(\'source_issue\')}\n                                hold = _hold(gate_row, 2_000_000)\n                                reason = \':\'.join(hold) if hold else None\n                                if leaf[\'format\'] == \'retained\' and any(str(t).upper() in (\'LOW\',\'QUARANTINE\')\n                                    for t in (extra.get(\'quality_tier\'),row.get(\'raw_quality_tier\'))):\n                                    reason = \'RETAINED_QUALITY_REVIEW\'\n                            if reason:\n                                held[reason] = held.get(reason,0)+1\n                                if raw_ok or not state[2]:\n                                    self.con.execute(\'UPDATE content SET reason=? WHERE fetch=? AND candidate IS NULL\', [reason,url])\n                                continue\n                            reference = {\'path\': \'parts/\'+identifier+\'.parquet\', \'row\':count,\'receipt_id\':identifier,\n                                         \'source_file_id\':remote[\'id\'], \'leaf\':leaf[\'name\'], \'original_snapshot_id\':original}\n                            candidates.append({\'fetch_url\':url,\'source_file_id\':remote[\'id\'],\'leaf\':leaf[\'name\'],\n                                \'original_snapshot_id\':original,\'source_text_sha256\':hashlib.sha256(row[\'text\'].encode()).hexdigest(),\n                                \'crawl_json\':row[\'crawl_json\'],\'extracted_json\':row[\'extracted_json\']})\n                            self.con.execute(\'UPDATE content SET candidate=?,reason=NULL WHERE fetch=?\', [json.dumps(reference),url])\n                            count += 1\n                        if candidates:\n                            writer.write_table(pa.Table.from_pylist(candidates,schema=schema))\n                        if time.monotonic()-logged>=15:\n                            print(\'COLLECT_PROGRESS\',leaf[\'name\'],\'| matched\',matched,\'| recovery candidates\',count,flush=True)\n                            logged=time.monotonic()\n                target = self.root/\'parts\'/(identifier+\'.parquet\')\n                if version_check is not None:\n                    version_check()\n                checksum = publish_file(output,target)\n                self.commit(identifier,\'raw_repair\' if repair else \'leaf\',{\'archive\':identity(remote),\'leaf\':leaf,\'candidates\':count,\n                    \'matched_rows\':matched,\'raw_rows\':raw_count,\'held\':held,\'path\':target.relative_to(self.root).as_posix(),\n                    \'sha256\':checksum,\'metadata_sha256\':staged_checksums,\'original_snapshot_provenance_preserved\':True,\n                    \'raw_path_policy\':RAW_PATH_POLICY,\'supersedes\':base_id if repair else None,\n                    \'row_scope\':\'successful_crawl_records_only\' if repair else \'full_source_frontier\'})\n                print(\'COLLECT_LEAF_SAVED:\', count, \'new candidate URLs |\', leaf[\'name\'], flush=True)\n            except BaseException:\n                self.con.rollback()\n                raise\n            finally:\n                con.close()\n\n    def pending_raw_repairs(self):\n        result=[]\n        for identifier,raw in self.con.execute("SELECT id,receipt FROM completed WHERE kind=\'leaf\' ORDER BY id"):\n            receipt=json.loads(raw)\n            if (receipt[\'leaf\'][\'format\']==\'native\' and receipt.get(\'raw_path_policy\')!=RAW_PATH_POLICY\n                    and not self.saved(repair_leaf_id(identifier))):\n                result.append((identifier,receipt))\n        return result\n\n    def _repair_metadata_index(self, *, refresh=False):\n        """Read only bounded metadata receipt patterns, never corpus/index payloads."""\n        leaves=[(r[0],json.loads(r[1])) for r in self.con.execute(\n            "SELECT id,receipt FROM completed WHERE kind=\'leaf\' ORDER BY id")]\n        binding=digest_json([(identifier,r[\'metadata_sha256\']) for identifier,r in leaves])\n        marker=self.root/\'raw_path_repair\'/\'metadata_index.json\'\n        if marker.is_file() and not refresh:\n            saved=checked(read_json(marker))\n            if saved[\'binding\']==binding:\n                print(\'REUSED_REPAIR_METADATA_INDEX | receipt JSON only\',flush=True)\n                return saved[\'paths\']\n        paths={}\n        print(\'REPAIR_METADATA_INDEX: saved receipt JSON only; no corpus/index/TAR scan\',flush=True)\n        def add(checksum,path):\n            if not isinstance(checksum,str) or not re.fullmatch(\'[0-9a-f]{64}\',checksum):\n                return\n            path=Path(path).resolve()\n            if not path.is_relative_to(self.project):\n                raise ValueError(\'Metadata reference escapes the project.\')\n            relative=path.relative_to(self.project).as_posix()\n            values=paths.setdefault(checksum,[])\n            if relative not in values:\n                values.append(relative)\n        for identifier,receipt in leaves:\n            for kind,checksum in receipt[\'metadata_sha256\'].items():\n                add(checksum,self.root/\'metadata_cache\'/identifier/kind/\'metadata.bin\')\n        patterns=(\n            \'data/processed/*/external_source/source.json\',\n            \'data_staging/btc-new-71be0e12-*/sources/*/metadata/metadata.json\',\n            \'data_staging/btc-new-71be0e12-*/*/sources/*/metadata/metadata.json\',\n            \'data_staging/btc-new-71be0e12-*/parallel_cpu4/jobs/*/sources/*/metadata/metadata.json\',\n        )\n        visited=set()\n        logged=time.monotonic()\n        for pattern in patterns:\n            for manifest in self.project.glob(pattern):\n                manifest=manifest.resolve()\n                if not manifest.is_relative_to(self.project):\n                    raise ValueError(\'Metadata receipt escapes the project.\')\n                if manifest in visited:\n                    continue\n                visited.add(manifest)\n                saved=read_json(manifest)\n                for item in saved.get(\'files\',{}).values():\n                    if isinstance(item,dict) and (item.get(\'path\') or item.get(\'name\')):\n                        add(item.get(\'sha256\'),manifest.parent/(item.get(\'path\') or item[\'name\']))\n                if time.monotonic()-logged>=15:\n                    print(\'REPAIR_METADATA_INDEX:\',len(visited),\'metadata receipts inspected\',flush=True)\n                    logged=time.monotonic()\n        atomic_json(marker,seal({\'binding\':binding,\'paths\':paths}))\n        print(\'REPAIR_METADATA_INDEX_READY:\',len(visited),\'saved metadata receipts; no corpus scan\',flush=True)\n        return paths\n\n    def _restore_repair_members(self, reader, identifier, receipt, missing, range_reads):\n        """Restore only absent, checksum-pinned member ranges; never scan a TAR."""\n        remote=receipt[\'archive\']\n        if identity(reader.metadata(remote[\'id\']))!=identity(remote):\n            raise ValueError(\'Original archive version changed; missing metadata was not requested.\')\n        references={}\n        with tempfile.TemporaryDirectory(prefix=\'repair-missing-\',dir=self.work) as temporary:\n            with reader.open(remote[\'id\'],remote) as stream:\n                for absent in missing:\n                    kind=absent[\'kind\']\n                    item=receipt[\'leaf\'][\'members\'][kind]\n                    checksum=receipt[\'metadata_sha256\'][kind]\n                    if (type(item[\'offset\']) is not int or type(item[\'bytes\']) is not int\n                            or item[\'offset\']<0 or not 0<item[\'bytes\']<=2*1024**3\n                            or item[\'offset\']+item[\'bytes\']>int(remote[\'size\'])\n                            or item.get(\'sha256\') not in (None,checksum)):\n                        raise ValueError(\'Saved member range/hash does not match the original source receipt.\')\n                    print(\'RESTORE_MISSING_METADATA_RANGE:\',receipt[\'leaf\'][\'name\'],\'|\',kind,\'|\',\n                          round(item[\'bytes\']/1024**2,1),\'MiB | no TAR scan or raw payload copy\',flush=True)\n                    range_reads.append({\'leaf_id\':identifier,\'kind\':kind,\'offset\':item[\'offset\'],\n                                        \'bytes\':item[\'bytes\'],\'sha256\':checksum})\n                    local=Path(temporary)/kind\n                    if isinstance(stream,CollectionDriveRange):\n                        stream.range_end=item[\'offset\']+item[\'bytes\']\n                    stage_member(stream,{**item,\'sha256\':checksum},local)\n                    if identity(reader.metadata(remote[\'id\']))!=identity(remote):\n                        raise ValueError(\'Archive version changed during metadata transfer; member not published.\')\n                    cache=self.root/\'metadata_cache\'/identifier/kind\n                    payload=cache/\'metadata.bin\'\n                    if publish_file(local,payload)!=checksum:\n                        raise ValueError(\'Restored metadata publication checksum mismatch.\')\n                    atomic_json(cache/\'receipt.json\',seal({\'binding\':digest_json(item),\n                        \'bytes\':item[\'bytes\'],\'sha256\':checksum}))\n                    references[kind]=(payload,checksum)\n                    print(\'METADATA_CHECKPOINT_SAVED_ON_DRIVE:\',kind,\'| exact original member restored\',flush=True)\n        return references\n\n    def repair_raw_paths(self, *, reader=None):\n        """Repair saved records, optionally restoring missing metadata member ranges.\n\n        Original receipts/candidate parts are retained. Each repaired leaf gets\n        its own durable SQL receipt and immutable recovery part; a new VM skips\n        committed repairs. By default missing metadata is reported, never fetched.\n        An explicit reader permits only missing, pinned metadata member ranges.\n        """\n        previous=checked(read_json(self.root/\'collection_report.json\'))\n        files=previous.get(\'archive_inventory\') or [json.loads(r[0])[\'remote\'] for r in self.con.execute(\n            "SELECT receipt FROM completed WHERE kind=\'archive\' ORDER BY id")]\n        if (previous[\'archive_errors\'] or len(files)!=previous[\'archive_files\']\n                or any(not self.saved(\'archive:\'+digest_json(identity(r))) for r in files)):\n            raise ValueError(\'Complete the source inventory before repairing raw classification.\')\n        pending=self.pending_raw_repairs()\n        if not pending and previous.get(\'raw_path_classification\',{}).get(\'state\')==\'CURRENT\':\n            print(\'RAW_PATH_REPAIR_ALREADY_COMPLETE | report only; no payload reads\',flush=True)\n            return previous\n        index=self._repair_metadata_index() if pending else {}\n        errors=[]\n        range_reads=[]\n        for number,(identifier,receipt) in enumerate(pending,1):\n            print(f\'RAW_PATH_REPAIR {number}/{len(pending)} | {receipt["leaf"]["name"]}\',flush=True)\n            references={}\n            missing=[]\n            for kind,item in receipt[\'leaf\'][\'members\'].items():\n                checksum=receipt[\'metadata_sha256\'][kind]\n                for relative in index.get(checksum,[]):\n                    path=(self.project/relative).resolve()\n                    if not path.is_relative_to(self.project):\n                        raise ValueError(\'Saved metadata index escapes the project.\')\n                    if path.is_file():\n                        references[kind]=(path,checksum)\n                        break\n                else:\n                    missing.append({\'kind\':kind,\'sha256\':checksum,\'bytes\':item[\'bytes\']})\n            try:\n                if missing and reader is not None:\n                    references.update(self._restore_repair_members(reader,identifier,receipt,missing,range_reads))\n                    missing=[]\n                if missing:\n                    errors.append({\'leaf\':receipt[\'leaf\'][\'name\'],\'leaf_id\':identifier,\'error\':\'SAVED_METADATA_MISSING\',\n                        \'missing\':missing,\'tar_download_attempted\':False})\n                    print(\'RAW_REPAIR_NEEDS_METADATA:\',receipt[\'leaf\'][\'name\'],\'| missing\',\n                          \',\'.join(r[\'kind\'] for r in missing),\'| no TAR reopened\',flush=True)\n                    continue\n                self.leaf(None,receipt[\'archive\'],receipt[\'leaf\'],repair=True,metadata_paths=references)\n            except Exception as error:\n                self.con.rollback()\n                errors.append({\'leaf\':receipt[\'leaf\'][\'name\'],\'leaf_id\':identifier,\'error\':str(error)[:500],\n                               \'tar_download_attempted\':any(r[\'leaf_id\']==identifier for r in range_reads)})\n        # A receipt may have committed locally before a transient snapshot write\n        # failed. Publish the final SQL snapshot before declaring completion.\n        self.snapshot()\n        remaining={identifier for identifier,_ in self.pending_raw_repairs()}\n        repair={\'state\':\'RAW_PATH_REPAIR_COMPLETE\' if not remaining else \'RAW_PATH_REPAIR_INCOMPLETE\',\n            \'policy\':RAW_PATH_POLICY,\'errors\':[e for e in errors if e[\'leaf_id\'] in remaining],\n            \'recovered_warnings\':[e for e in errors if e[\'leaf_id\'] not in remaining],\n            \'tar_accessed\':bool(range_reads),\'metadata_range_reads\':range_reads,\'updated_at\':utc_now()}\n        atomic_json(self.root/\'raw_path_repair\'/\'repair_report.json\',seal(repair))\n        return self.report(files,[],raw_repair=repair)\n\n    def report(self, files, errors, *, raw_repair=None):\n        # Count official IDs and distinct fetch URLs separately. Failed records never count as usable text.\n        state = "CASE WHEN c.existing IS NOT NULL THEN \'REUSED_VALIDATED_CONTENT\' WHEN c.candidate IS NOT NULL THEN \'RECOVERED_EXTRACTION_PENDING_CHUNK\' WHEN c.seen=0 THEN \'NOT_FOUND_IN_ARCHIVES\' WHEN c.raw=1 THEN \'RAW_AVAILABLE_NEEDS_EXTRACTION\' WHEN c.raw_status=1 THEN \'RAW_METADATA_REVIEW\' ELSE \'CRAWL_NOT_SUCCESSFUL\' END"\n        query = f\'SELECT o.id,o.url,o.fetch,{state} AS state,c.existing,c.candidate,c.reason FROM official o JOIN content c ON o.fetch=c.fetch\'\n        counts = dict(self.con.execute(f\'SELECT state,count(*) FROM ({query}) GROUP BY state\'))\n        total = self.con.execute(\'SELECT count(*) FROM official\').fetchone()[0]\n        unique = self.con.execute(\'SELECT count(*) FROM content\').fetchone()[0]\n        seen,raw=self.con.execute(\'SELECT sum(c.seen),sum(c.raw) FROM official o JOIN content c ON o.fetch=c.fetch\').fetchone()\n        files_out={}\n        for name, sql in [(\'btc_membership.csv\',query),(\'missing_or_unusable.csv\',f"SELECT * FROM ({query}) WHERE state NOT IN (\'REUSED_VALIDATED_CONTENT\',\'RECOVERED_EXTRACTION_PENDING_CHUNK\')")]:\n            import csv\n            path = self.work/name\n            cursor = self.con.execute(sql)\n            with path.open(\'w\',encoding=\'utf-8\',newline=\'\') as out:\n                writer = csv.writer(out)\n                writer.writerow([c[0] for c in cursor.description])\n                while rows := cursor.fetchmany(8192):\n                    writer.writerows(rows)\n            files_out[name]={\'path\':name,\'sha256\':publish_file(path,self.root/name)}\n        pending_repairs=len(self.pending_raw_repairs())\n        recovery_parts=[]\n        for identifier,kind,raw_receipt in self.con.execute(\n                "SELECT id,kind,receipt FROM completed WHERE kind IN (\'leaf\',\'raw_repair\')"):\n            receipt=json.loads(raw_receipt)\n            superseded=(kind==\'leaf\' and receipt[\'leaf\'][\'format\']==\'native\'\n                        and self.saved(repair_leaf_id(identifier)) is not None)\n            recovery_parts.append({**receipt,\'classification_effective\':not superseded,\n                \'superseded_classification_by\':repair_leaf_id(identifier) if superseded else None})\n        result = seal({\'state\':\'ALL_FOLDER_SOURCES_COLLECTED\' if not errors else \'FOLDER_COLLECTION_INCOMPLETE\',\n            \'official_ids\':total,\'official_unique_fetch_urls\':unique,\'counts_by_official_id\':counts,\n            \'official_ids_seen_in_archive_metadata\':seen,\'official_ids_with_successful_raw_record\':raw,\n            \'all_official_ids_represented_in_archives\':not errors and seen==total,\n            \'all_official_raw_records_successful\':not errors and not pending_repairs and raw==total,\n            \'archive_files\':len(files),\'archive_inventory\':files,\'archive_errors\':errors,\'files\':files_out,\n            \'recovery_parts\':recovery_parts,\n            \'all_official_content_available\':not errors and not pending_repairs and sum(counts.get(k,0) for k in (\'REUSED_VALIDATED_CONTENT\',\'RECOVERED_EXTRACTION_PENDING_CHUNK\'))==total,\n            \'raw_path_classification\':{\'policy\':RAW_PATH_POLICY,\'state\':\'REPAIR_REQUIRED\' if pending_repairs else \'CURRENT\',\n                                       \'pending_leaves\':pending_repairs},\n            \'counts_provisional\':bool(pending_repairs),\'raw_path_repair\':raw_repair,\n            \'recovered_text_requires_standard_chunk_pipeline\':True,\'raw_payloads_rehashed\':False,\n            \'submission_candidate_changed\':False,\'updated_at\':utc_now()})\n        atomic_json(self.root/\'collection_report.json\',result)\n        return result\n\n    def run(self, reader, files, retained):\n        errors = []\n        for index, remote in enumerate(files,1):\n            print(f\'ALL_AI_GURU {index}/{len(files)} | {remote["name"]}\',flush=True)\n            archive_id = \'archive:\'+digest_json(identity(remote))\n            if self.saved(archive_id):\n                print(\'SKIP_COMPLETED_ARCHIVE | durable SQL receipt; no TAR reopened\',flush=True)\n                continue\n            try:\n                def version_check():\n                    if identity(reader.metadata(remote[\'id\'])) != identity(remote):\n                        raise ValueError(\'Archive changed during collection; this leaf was not committed.\')\n                with metadata_leaves(reader,remote,retained,self.work) as (stream,leaves):\n                    for leaf in leaves:\n                        self.leaf(stream,remote,leaf,version_check=version_check)\n                if identity(reader.metadata(remote[\'id\'])) != identity(remote):\n                    raise ValueError(\'Archive changed during collection.\')\n                self.commit(archive_id,\'archive\',{\'remote\':identity(remote),\'leaves\':len(leaves)})\n            except Exception as error:\n                self.con.rollback()\n                record={\'file\':remote[\'name\'],\'id\':remote[\'id\'],\'error\':str(error)[:500]}\n                if isinstance(error,CollectionDriveError):\n                    record[\'drive_error\']=error.details\n                errors.append(record)\n                print(\'ARCHIVE_NEEDS_INSPECTION:\',remote[\'name\'],type(error).__name__,str(error)[:200],flush=True)\n                if isinstance(error,CollectionDriveError) and error.details[\'pause_campaign\']:\n                    print(\'DRIVE_COOLDOWN: campaign paused; remaining archives not requested. Resume same checkpoint later.\',flush=True)\n                    errors.extend({\'file\':r[\'name\'],\'id\':r[\'id\'],\n                        \'error\':\'NOT_ATTEMPTED_DRIVE_COOLDOWN\',\'blocked_by\':remote[\'id\']}\n                        for r in files[index:] if not self.saved(\'archive:\'+digest_json(identity(r))))\n                    break\n        return self.report(files,errors)\n\n    def close(self):\n        self.con.close()\n'
MAXIMIZE_SOURCE='"""CPU-only delta recovery. Paid corpus and active submission remain immutable."""\nfrom collections import Counter\nfrom dataclasses import asdict\nimport gzip\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport shutil\nimport sqlite3\nimport tarfile\nimport tempfile\nimport time\n\nimport duckdb\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\ntry:\n    from collect_all_btc_sources import CollectAll, identity, raw_record_problem\nexcept ModuleNotFoundError:\n    from scripts.collect_all_btc_sources import CollectAll, identity, raw_record_problem\nfrom vietmedbridge.artifacts import atomic_json,code_fingerprint,digest_json,publish_file,read_json,runtime_versions,utc_now,verify_file\nfrom vietmedbridge.content_embeddings import checked,seal\nfrom vietmedbridge.build import verify_spans\nfrom vietmedbridge.chunks import chunk_source\nfrom vietmedbridge.external_import import SCHEMAS,_document,_hold\nfrom vietmedbridge.isolated_intake import Window\nfrom vietmedbridge.quality import document_quality\nfrom vietmedbridge.structure import parse_structure\nfrom vietmedbridge.text import EXPECTED_PARSE_ERRORS,extract_source,normalize_for_retrieval\nfrom vietmedbridge.validation import SCHEMA_VERSION,validate_snapshot\nfrom vietmedbridge.health import health_report,freeze_candidate\nfrom vietmedbridge.index_inputs import prepare_index_inputs\n\nPOLICY=\'collected-delta-raw-recovery-v1\'\nCAMPAIGN=\'btc-maximize-71be0e12-v1\'\nMAX_BODY=64*1024**2\nRECOVERY_SCHEMA=pa.schema([(\'fetch_url\',pa.string()),(\'source_file_id\',pa.string()),(\'leaf\',pa.string()),\n    (\'original_snapshot_id\',pa.string()),(\'source_text_sha256\',pa.string()),(\'crawl_json\',pa.string()),\n    (\'extracted_json\',pa.large_string())])\n\n\ndef inside(root,relative):\n    path=(Path(root)/relative).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError(\'Recovery artifact escapes its root.\')\n    return path\n\n\ndef unpack_raw(payload,path,record):\n    """Check compressed OR decoded SHA/length explicitly; never guess the convention."""\n    size=record.get(\'raw_size_bytes\')\n    if type(size) is not int or not 0<size<=MAX_BODY:\n        raise ValueError(\'RAW_SIZE_REQUIRES_REVIEW\')\n    if path.endswith(\'.zst\'):\n        # zstd frame content size may be absent: use a bounded streaming decoder.\n        with pa.CompressedInputStream(pa.BufferReader(payload),\'zstd\') as stream:\n            body=stream.read(MAX_BODY+1)\n    elif path.endswith(\'.gz\'):\n        with gzip.GzipFile(fileobj=io.BytesIO(payload)) as stream:\n            body=stream.read(MAX_BODY+1)\n    else:\n        body=payload\n    if len(body)>MAX_BODY:\n        raise ValueError(\'RAW_SIZE_REQUIRES_REVIEW\')\n    expected=record[\'raw_sha256\']\n    if len(body)==size and hashlib.sha256(body).hexdigest()==expected:\n        convention=\'decoded-body\'\n    elif len(payload)==size and hashlib.sha256(payload).hexdigest()==expected:\n        convention=\'stored-payload\'\n    else:\n        raise ValueError(\'RAW_PAYLOAD_HASH_OR_LENGTH_MISMATCH\')\n    return body,convention\n\n\nclass MaximizeCollection:\n    snapshot=CollectAll.snapshot\n    saved=CollectAll.saved\n    commit=CollectAll.commit\n\n    def __init__(self,collection,work,*,tokenizer_spec,chunk_config):\n        self.collection=collection\n        self.project=collection.project\n        self.root=self.project/\'data_staging\'/CAMPAIGN\n        self.work=Path(work)\n        self.root.mkdir(parents=True,exist_ok=True)\n        self.work.mkdir(parents=True,exist_ok=True)\n        self.report=checked(read_json(collection.root/\'collection_report.json\'))\n        if self.report[\'counts_provisional\'] or self.report[\'archive_errors\']:\n            raise ValueError(\'Complete collection/classification before delta recovery.\')\n        self.binding={\'policy\':POLICY,\'collection\':digest_json(self.report),\n            \'tokenizer\':tokenizer_spec,\'chunks\':asdict(chunk_config),\n            \'producer_code\':code_fingerprint(),\'runtime\':runtime_versions()}\n        self.tokenizer_spec,self.chunk_config=tokenizer_spec,chunk_config\n        self.pointer=self.root/\'checkpoint.json\'\n        self.database=self.work/\'collection.sqlite\'\n        if not self.database.is_file() and self.pointer.is_file():\n            saved=checked(read_json(self.pointer))\n            if saved[\'binding\']!=self.binding:\n                raise ValueError(\'Recovery inputs/policy changed; preserve this campaign.\')\n            source=inside(self.root,saved[\'path\'])\n            verify_file(source,saved[\'sha256\'])\n            print(\'RESTORE_MAXIMIZE: one SQL snapshot; completed sources not replayed\',flush=True)\n            temporary=self.database.with_suffix(\'.restore.partial\')\n            try:\n                with gzip.open(source,\'rb\') as incoming,temporary.open(\'wb\') as out:\n                    shutil.copyfileobj(incoming,out,1024**2)\n                verify_file(temporary,saved[\'sqlite_sha256\'])\n                temporary.replace(self.database)\n            finally:\n                temporary.unlink(missing_ok=True)\n        self.con=sqlite3.connect(self.database)\n        self.con.execute(\'PRAGMA cache_size=-32768\')\n        self.con.execute(\'PRAGMA synchronous=FULL\')\n        self.con.executescript(\'\'\'CREATE TABLE IF NOT EXISTS contract(value TEXT);\n            CREATE TABLE IF NOT EXISTS completed(id TEXT PRIMARY KEY,kind TEXT,receipt TEXT);\n            CREATE TABLE IF NOT EXISTS aliases(id INTEGER PRIMARY KEY,url TEXT,fetch TEXT);\n            CREATE INDEX IF NOT EXISTS aliases_fetch ON aliases(fetch);\n            CREATE TABLE IF NOT EXISTS todo(fetch TEXT PRIMARY KEY,state TEXT,candidate TEXT,reason TEXT);\n            CREATE TABLE IF NOT EXISTS tasks(leaf_id TEXT,fetch TEXT,raw_path TEXT,sha TEXT,crawl TEXT,\n                quality_review INTEGER,PRIMARY KEY(leaf_id,fetch));\n            CREATE TABLE IF NOT EXISTS raw_results(sha TEXT,source_url TEXT,quality_review INTEGER,result TEXT,\n                PRIMARY KEY(sha,source_url,quality_review));\n        \'\'\')\n        row=self.con.execute(\'SELECT value FROM contract\').fetchone()\n        if row and row[0]!=digest_json(self.binding):\n            raise ValueError(\'Local recovery SQL belongs to another frozen input.\')\n        if not row:\n            self.con.execute(\'INSERT INTO contract VALUES (?)\',[digest_json(self.binding)])\n        self.con.commit()\n        if not self.saved(\'seed\'):\n            # URL membership and pointers only. Existing document/vector payloads are never opened.\n            cursor=collection.con.execute(\'\'\'SELECT o.id,o.url,o.fetch,c.candidate,c.reason,c.raw\n                FROM official o JOIN content c ON o.fetch=c.fetch\n                WHERE c.existing IS NULL\'\'\')\n            seeded=0; logged=time.monotonic()\n            for rows in iter(lambda:cursor.fetchmany(2048),[]):\n                for doc_id,url,fetch,candidate,reason,raw in rows:\n                    self.con.execute(\'INSERT OR IGNORE INTO aliases VALUES (?,?,?)\',[doc_id,url,fetch])\n                    state=\'TEXT_READY\' if candidate else \'NEEDS_RAW\' if raw else \'NEEDS_CRAWL\'\n                    self.con.execute(\'INSERT OR IGNORE INTO todo VALUES (?,?,?,?)\',[fetch,state,candidate,reason])\n                seeded+=len(rows)\n                if time.monotonic()-logged>=15:\n                    print(\'DELTA_MEMBERSHIP:\',f\'{seeded:,}\',\'official IDs | pointers only; no corpus/vector reads\',flush=True)\n                    logged=time.monotonic()\n            self.commit(\'seed\',\'seed\',{\'counts\':self.report[\'counts_by_official_id\']})\n\n    def close(self):\n        if getattr(self,\'_candidate_db\',None) is not None:\n            self._candidate_db.close()\n            self._candidate_db=None\n        self.con.close()\n\n    def status(self):\n        counts=dict(self.con.execute(\'\'\'SELECT t.state,count(*) FROM aliases a JOIN todo t USING(fetch)\n            GROUP BY t.state\'\'\'))\n        result=seal({\'policy\':POLICY,\'state\':\'DELTA_RECOVERY_STATUS\',\'counts_by_official_id\':counts,\n            \'reused_validated_ids\':self.report[\'counts_by_official_id\'].get(\'REUSED_VALIDATED_CONTENT\',0),\n            \'raw_pass_complete\':self.saved(\'raw-pass\') is not None,\n            \'raw_source_errors\':self.source_errors,\'submission_candidate_changed\':False,\n            \'scope\':\'CPU extraction/chunks only; vectors and submission are separate\',\'updated_at\':utc_now()})\n        atomic_json(self.root/\'status.json\',result)\n        print(\'DELTA_STATUS:\',json.dumps(counts,ensure_ascii=False),flush=True)\n        return result\n\n    @property\n    def source_errors(self):\n        return getattr(self,\'_source_errors\',[])\n\n    def _candidate(self,reference):\n        root=self.root if reference.get(\'recovery_root\')==\'maximize\' else self.collection.root\n        item=reference\n        if \'sha256\' not in item:\n            receipt=self.collection.saved(item[\'receipt_id\'])\n            item={**item,\'sha256\':receipt[\'sha256\']}\n        if getattr(self,\'_candidate_db\',None) is None:\n            self._candidate_db=sqlite3.connect(self.work/\'candidate_rows.sqlite\')\n            self._candidate_db.execute(\'PRAGMA cache_size=-32768\')\n            self._candidate_db.executescript(\'\'\'CREATE TABLE IF NOT EXISTS parts(sha TEXT PRIMARY KEY,rows INTEGER);\n                CREATE TABLE IF NOT EXISTS records(sha TEXT,position INTEGER,record TEXT,PRIMARY KEY(sha,position));\'\'\')\n        cache=self._candidate_db\n        sha=item[\'sha256\']\n        if cache.execute(\'SELECT 1 FROM parts WHERE sha=?\',[sha]).fetchone() is None:\n            local=self.work/\'candidate_cache\'/sha\n            if local.is_file():\n                verify_file(local,sha)\n            else:\n                if publish_file(inside(root,item[\'path\']),local)!=sha:\n                    raise ValueError(\'Recovery candidate checksum changed.\')\n            position=0\n            try:\n                # Decode each small delta part once, with bounded batches and a disk row index.\n                for batch in pq.ParquetFile(local).iter_batches(batch_size=128):\n                    rows=[(sha,position+i,json.dumps(row,ensure_ascii=False)) for i,row in enumerate(batch.to_pylist())]\n                    cache.executemany(\'INSERT INTO records VALUES (?,?,?)\',rows)\n                    position+=len(rows)\n                cache.execute(\'INSERT INTO parts VALUES (?,?)\',[sha,position]); cache.commit()\n            except BaseException:\n                cache.rollback(); raise\n        row=cache.execute(\'SELECT record FROM records WHERE sha=? AND position=?\',[sha,item[\'row\']]).fetchone()\n        if row is None: raise ValueError(\'Candidate row exceeds its receipt.\')\n        return json.loads(row[0])\n\n    def chunk_ready(self,tokenizer,*,phase):\n        """Normal source spans/BGE token policy; one durable delta shard per 128 URLs."""\n        if phase not in (\'metadata-text-v1\',\'raw-text-v1\'):\n            raise ValueError(\'Use separate immutable phases for collected text and raw recovery.\')\n        origin="json_extract(candidate,\'$.recovery_root\')=\'maximize\'" if phase==\'raw-text-v1\' else "coalesce(json_extract(candidate,\'$.recovery_root\'),\'\')!=\'maximize\'"\n        target=self.root/\'processed\'/phase\n        target.mkdir(parents=True,exist_ok=True)\n        config={\'schema_version\':SCHEMA_VERSION,\'code_sha256\':code_fingerprint(),\n            \'official_links_sha256\':self.collection.binding[\'official_sha256\'],\n            \'tokenizer\':self.tokenizer_spec,\'chunks\':asdict(self.chunk_config),\n            \'recovery_binding\':self.binding,\'phase\':phase}\n        signature=digest_json(config)\n        atomic_json(target/\'config.json\',{**config,\'signature\':signature})\n        while rows:=self.con.execute("SELECT fetch,candidate FROM todo WHERE state=\'TEXT_READY\' AND "+origin+" ORDER BY fetch LIMIT 128").fetchall():\n            key=\'chunk:\'+digest_json({\'phase\':phase,\'rows\':rows,\'signature\':signature})\n            saved=self.saved(key)\n            if saved:\n                raise ValueError(\'Chunk receipt exists but its SQL outcomes were not committed together.\')\n            count=Counter()\n            with tempfile.TemporaryDirectory(prefix=\'delta-chunk-\',dir=self.work) as temporary:\n                local=Path(temporary)\n                paths={k:local/(k+\'.parquet\') for k in SCHEMAS}\n                writers={k:pq.ParquetWriter(paths[k],s,compression=\'zstd\') for k,s in SCHEMAS.items()}\n                outcomes=[]\n                try:\n                    for fetch,reference in rows:\n                        source=self._candidate(json.loads(reference))\n                        if source[\'fetch_url\']!=fetch:\n                            raise ValueError(\'Candidate URL differs from official membership.\')\n                        crawl=json.loads(source[\'crawl_json\']); extra=json.loads(source[\'extracted_json\'])\n                        text=extra.get(\'text\') or \'\'\n                        if (hashlib.sha256(text.encode()).hexdigest()!=source[\'source_text_sha256\']\n                                or crawl.get(\'fetch_url\')!=fetch\n                                or crawl[\'snapshot_id\']!=source[\'original_snapshot_id\']\n                                or extra[\'snapshot_id\']!=crawl[\'snapshot_id\']\n                                or extra[\'raw_sha256\']!=crawl[\'raw_sha256\'] or len(text)!=extra[\'char_count\']):\n                            raise ValueError(\'Recovered extraction provenance/length changed.\')\n                        success=True\n                        for doc_id,url in self.con.execute(\'SELECT id,url FROM aliases WHERE fetch=? ORDER BY id\',[fetch]):\n                            row={**crawl,**extra,\'status\':crawl[\'status\'],\'url\':url}\n                            row.update(language=extra.get(\'language\'),language_method=extra.get(\'language_method\'),\n                                language_confidence=extra.get(\'language_confidence\'),extractor=extra.get(\'extractor\'),\n                                extractor_version=extra.get(\'extractor_version\'))\n                            hold=_hold(row,2_000_000)\n                            if hold:\n                                raise ValueError(\'A previously accepted candidate now fails its extraction gate: \'+str(hold))\n                            row[\'doc_id\']=doc_id\n                            doc=_document(row,source[\'source_file_id\']+\'!\'+source[\'leaf\'])\n                            structure=parse_structure(doc,doc[\'heading_hints\'])\n                            doc.update(document_quality(text,title=doc[\'title\'],section_count=structure[\'known_heading_count\'],\n                                raw_bytes=crawl.get(\'raw_size_bytes\')),section_count=structure[\'known_heading_count\'])\n                            content_hash=hashlib.sha256(normalize_for_retrieval(text).encode()).hexdigest()\n                            doc.update(content_hash=content_hash,canonical_content_id=content_hash)\n                            ledger={\'doc_id\':doc_id,\'url\':url,\'domain\':doc[\'domain\'],\'status\':\'ok\',\n                                \'content_type\':doc[\'content_type\'],\'raw_bytes\':crawl.get(\'raw_size_bytes\') or 0,\n                                \'elapsed_ms\':None,\'http_status\':crawl.get(\'http_status\')}\n                            tables={\'ledger\':[ledger]}\n                            count[\'input_records\']+=1\n                            try:\n                                children,parents=chunk_source(doc,tokenizer,self.tokenizer_spec,self.chunk_config,structure=structure)\n                                verify_spans(doc,children,parents)\n                                if not children: raise ValueError(\'NO_INDEXABLE_CHILDREN\')\n                                tables.update(documents=[doc],children=children,parents=parents,sections=structure[\'sections\'])\n                                for kind in (\'documents\',\'children\',\'parents\',\'sections\'): count[kind]+=len(tables[kind])\n                            except ValueError as error:\n                                success=False\n                                tables[\'failures\']=[{\'doc_id\':doc_id,\'url\':url,\'phase\':\'chunk\',\'reason\':str(error),\n                                    \'reason_code\':\'CHUNK_REVIEW\',\'body_sha256\':doc[\'body_sha256\'],\'raw_file\':doc[\'raw_file\'],\n                                    \'domain\':doc[\'domain\'],\'content_type\':doc[\'content_type\']}]\n                                count[\'failures\']+=1\n                            for kind,values in tables.items():\n                                writers[kind].write_table(pa.Table.from_pylist(values,schema=SCHEMAS[kind]))\n                        outcomes.append((\'CHUNKED\' if success else \'CHUNK_REVIEW\',fetch))\n                finally:\n                    for writer in writers.values(): writer.close()\n                files={}\n                for kind,path in paths.items():\n                    relative=\'parts/\'+key[6:]+\'/\'+kind+\'.parquet\'\n                    files[kind]={\'path\':relative,\'sha256\':publish_file(path,target/relative)}\n            self.con.executemany(\'UPDATE todo SET state=? WHERE fetch=?\',outcomes)\n            self.commit(key,\'chunk\',{\'phase\':phase,\'signature\':signature,\'files\':files,\'counts\':dict(count),\n                \'records\':count[\'input_records\'],\'completed_at\':utc_now()})\n            print(\'DELTA_CHUNK_SAVED:\',phase,\'|\',count[\'documents\'],\'documents | existing corpus untouched\',flush=True)\n        parts=[json.loads(r[0]) for r in self.con.execute("SELECT receipt FROM completed WHERE kind=\'chunk\' ORDER BY id")\n               if json.loads(r[0])[\'phase\']==phase]\n        if not parts: return None\n        counts=Counter()\n        for p in parts: counts.update(p[\'counts\'])\n        build={\'schema_version\':SCHEMA_VERSION,\'signature\':signature,\'parts\':parts,\'counts\':dict(counts),\n            \'run_name\':phase,\'selected_range_complete\':True,\'selected_shards\':len(parts),\n            \'chunking\':asdict(self.chunk_config),\n            \'requested_input_records\':counts[\'input_records\'],\'input_kind\':\'COLLECTED_RECOVERY_DELTA\',\n            \'state\':\'DATA_VALIDATED\',\'snapshot_sha256\':digest_json({\'signature\':signature,\'parts\':parts})}\n        atomic_json(target/\'build.json\',build)\n        return target,build\n\n    def prepare_raw_tasks(self,reader):\n        # Each native source version is inspected once, using already cached metadata.\n        originals=[(r[0],json.loads(r[1])) for r in self.collection.con.execute(\n            "SELECT id,receipt FROM completed WHERE kind=\'leaf\' ORDER BY id")]\n        originals=[(identifier,receipt) for identifier,receipt in originals if not self.saved(\'tasks:\'+identifier)]\n        if not originals:\n            print(\'RAW_TASKS_REUSED: receipt inventory only; no metadata payload replay\',flush=True)\n            return\n        index=self.collection._repair_metadata_index()\n        for identifier,receipt in originals:\n            key=\'tasks:\'+identifier\n            if self.saved(key): continue\n            leaf=receipt[\'leaf\']; members=leaf[\'members\']\n            print(\'RAW_TASK_SOURCE_START:\',leaf[\'name\'],\'| saved metadata only\',flush=True)\n            native=leaf[\'format\']==\'native\'\n            kinds=(\'dataset\',\'frontier\',\'crawl\') if native else (\'dataset\',\'raw_catalog\')\n            with tempfile.TemporaryDirectory(prefix=\'raw-task-\',dir=self.work) as temporary:\n                local=Path(temporary)\n                for kind in kinds:\n                    sha=receipt[\'metadata_sha256\'][kind]\n                    source=next((inside(self.project,p) for p in index.get(sha,[]) if inside(self.project,p).is_file()),None)\n                    if source is None:\n                        restored=self.collection._restore_repair_members(reader,identifier,receipt,[{\'kind\':kind}],[])\n                        source=restored[kind][0]\n                    destination=local/kind\n                    if publish_file(source,destination)!=sha: raise ValueError(\'Task metadata checksum mismatch.\')\n                descriptor=read_json(local/\'dataset\')\n                con=duckdb.connect()\n                try:\n                    con.execute("SET memory_limit=\'1GB\'"); con.execute(\'SET threads=1\')\n                    con.execute(\'SET preserve_insertion_order=false\')\n                    con.execute(\'SET temp_directory=?\',[str(local/\'spill\')])\n                    if native:\n                        con.read_parquet(str(local/\'frontier\')).create_view(\'f\')\n                        con.read_parquet(str(local/\'crawl\')).create_view(\'c\')\n                        query="SELECT f.fetch_url,to_json(c) AS crawl,NULL::VARCHAR AS quality FROM f JOIN c USING(crawl_url_id) WHERE c.status=\'SUCCESS_RAW\'"\n                    else:\n                        con.read_parquet(str(local/\'raw_catalog\')).create_view(\'r\')\n                        query=\'\'\'SELECT fetch_url,original_crawl_record_json AS crawl,quality_tier AS quality,\n                            raw_path,raw_sha256,original_snapshot_id FROM r\'\'\'\n                    tasks=0\n                    for batch in con.execute(query).fetch_record_batch(256):\n                        for item in batch.to_pylist():\n                            fetch=item[\'fetch_url\']\n                            state=self.con.execute(\'SELECT state FROM todo WHERE fetch=?\',[fetch]).fetchone()\n                            if not state or state[0]!=\'NEEDS_RAW\': continue\n                            crawl=json.loads(item[\'crawl\'])\n                            if crawl.get(\'fetch_url\')!=fetch:\n                                raise ValueError(\'Raw crawl URL differs from native/BTC membership.\')\n                            locator=crawl if native else {**crawl,\'raw_path\':item[\'raw_path\']}\n                            if (not native and (item[\'raw_sha256\']!=crawl.get(\'raw_sha256\')\n                                    or item[\'original_snapshot_id\']!=crawl.get(\'snapshot_id\'))):\n                                raise ValueError(\'Retained locator and original crawl binding differ.\')\n                            if raw_record_problem({**locator,\'crawl_snapshot\':crawl.get(\'snapshot_id\')},leaf[\'format\'],descriptor): continue\n                            self.con.execute(\'INSERT OR IGNORE INTO tasks VALUES (?,?,?,?,?,?)\',\n                                [identifier,fetch,locator[\'raw_path\'],crawl[\'raw_sha256\'],item[\'crawl\'],\n                                 int(str(item[\'quality\']).upper() in (\'LOW\',\'QUARANTINE\'))])\n                            tasks+=1\n                finally: con.close()\n            self.commit(key,\'tasks\',{\'leaf_id\':identifier,\'tasks\':tasks})\n            print(\'RAW_TASK_SOURCE_SAVED:\',leaf[\'name\'],\'|\',tasks,\'raw versions; completed source skipped on resume\',flush=True)\n\n    def _save_extracted(self,source,rows):\n        if not rows: return\n        key=digest_json(rows)\n        with tempfile.TemporaryDirectory(dir=self.work) as temporary:\n            path=Path(temporary)/\'recovered.parquet\'\n            pq.write_table(pa.Table.from_pylist(rows,schema=RECOVERY_SCHEMA),path,compression=\'zstd\')\n            relative=\'recovered/\'+key+\'.parquet\'\n            sha=publish_file(path,self.root/relative)\n        for number,row in enumerate(rows):\n            ref={\'path\':relative,\'row\':number,\'sha256\':sha,\'recovery_root\':\'maximize\',\n                 \'source_file_id\':source[\'archive\'][\'id\'],\'original_snapshot_id\':row[\'original_snapshot_id\']}\n            self.con.execute("UPDATE todo SET state=\'TEXT_READY\',candidate=?,reason=NULL WHERE fetch=? AND state=\'NEEDS_RAW\'",\n                [json.dumps(ref),row[\'fetch_url\']])\n\n    def recover_raw(self,reader,*,checkpoint_headers=1024):\n        """One bounded TAR header pass per needed source; resume at saved next header."""\n        if self.saved(\'raw-pass\'):\n            print(\'RAW_PASS_REUSED: completed recovery checkpoint; no source/payload reads\',flush=True)\n            return self.status()\n        self.prepare_raw_tasks(reader)\n        self._source_errors=[]\n        sources=[(r[0],json.loads(r[1])) for r in self.collection.con.execute(\n            "SELECT id,receipt FROM completed WHERE kind=\'leaf\' ORDER BY id")]\n        for leaf_id,receipt in sources:\n            if self.saved(\'raw:\'+leaf_id): continue\n            tasks=self.con.execute("""SELECT t.fetch,t.raw_path,t.sha,t.crawl,t.quality_review FROM tasks t\n                JOIN todo o USING(fetch) WHERE t.leaf_id=? AND o.state=\'NEEDS_RAW\' AND NOT EXISTS (\n                SELECT 1 FROM raw_results r WHERE r.sha=t.sha AND r.source_url=t.fetch\n                AND r.quality_review=t.quality_review)""",[leaf_id]).fetchall()\n            if not tasks: continue\n            remote=receipt[\'archive\']\n            if identity(reader.metadata(remote[\'id\']))!=identity(remote):\n                raise ValueError(\'Raw archive changed since collection: \'+remote[\'name\'])\n            by_path={}\n            for task in tasks: by_path.setdefault(task[1],[]).append(task)\n            remaining_paths=set(by_path)\n            cursor_key=\'cursor:\'+leaf_id\n            state=self.saved(cursor_key) or {\'offset\':0,\'leaf_start\':None,\'leaf_size\':None,\'pax\':{}}\n            if state.get(\'eof\'):\n                self.commit(\'raw:\'+leaf_id,\'raw_source\',{\'leaf_id\':leaf_id,\'complete\':True})\n                continue\n            # Find a nested leaf via TAR headers only, keeping payload reads bounded.\n            with reader.open(remote[\'id\'],remote) as stream:\n                if hasattr(stream,\'block_size\'): stream.block_size=8*1024**2\n                start,size=0,int(remote[\'size\'])\n                if state[\'leaf_start\'] is not None:\n                    start,size=state[\'leaf_start\'],state[\'leaf_size\']\n                else:\n                    for child in receipt[\'leaf\'][\'name\'].split(\'!\')[1:]:\n                        with tarfile.open(fileobj=Window(stream,start,size),mode=\'r:\') as archive:\n                            member=next((m for m in archive if m.name==child and m.isfile()),None)\n                            if member is None: raise ValueError(\'Nested raw leaf missing: \'+child)\n                            start,size=start+member.offset_data,member.size\n                offset=state[\'offset\']; pending=[]; headers=0; logged=time.monotonic()\n                if offset==size:\n                    self.commit(\'raw:\'+leaf_id,\'raw_source\',{\'leaf_id\':leaf_id,\'complete\':True})\n                    continue\n                stream.seek(start+offset)\n                boundary=stream.read(min(512,size-offset))\n                if boundary and not any(boundary):\n                    self.commit(\'raw:\'+leaf_id,\'raw_source\',{\'leaf_id\':leaf_id,\'complete\':True})\n                    continue\n                print(\'RAW_SOURCE_RESUME:\',receipt[\'leaf\'][\'name\'],\'| header byte\',offset,\'| pending URLs\',len(tasks),flush=True)\n                def save(next_offset,*,eof=False):\n                    self._save_extracted(receipt,pending); pending.clear()\n                    value={\'offset\':next_offset,\'leaf_start\':start,\'leaf_size\':size,\'pax\':dict(archive.pax_headers),\'eof\':eof}\n                    self.con.execute(\'INSERT OR REPLACE INTO completed VALUES (?,?,?)\',\n                        [cursor_key,\'cursor\',json.dumps(value)])\n                    self.con.commit(); self.snapshot()\n                    print(\'RAW_CURSOR_SAVED_ON_DRIVE:\',receipt[\'leaf\'][\'name\'],\'| next header\',next_offset,flush=True)\n                with tarfile.open(fileobj=Window(stream,start+offset,size-offset),mode=\'r:\') as archive:\n                    archive.pax_headers.update(state.get(\'pax\',{}))\n                    for member in archive:\n                        headers+=1\n                        if member.isfile() and member.name in by_path:\n                            needed=[t for t in by_path[member.name] if self.con.execute(\'SELECT state FROM todo WHERE fetch=?\',[t[0]]).fetchone()[0]==\'NEEDS_RAW\']\n                            if needed:\n                                if not 0<member.size<=MAX_BODY:\n                                    for task in needed: self._raw_result(task,\'RAW_SIZE_REQUIRES_REVIEW\')\n                                else:\n                                    payload=archive.extractfile(member).read(MAX_BODY+1)\n                                    for task in needed:\n                                        fetch,path,sha,raw,quality_review=task\n                                        if self.con.execute(\'SELECT 1 FROM raw_results WHERE sha=? AND source_url=? AND quality_review=?\',\n                                                [sha,fetch,quality_review]).fetchone(): continue\n                                        crawl=json.loads(raw)\n                                        try:\n                                            body,convention=unpack_raw(payload,path,crawl)\n                                            if quality_review: raise ValueError(\'RETAINED_QUALITY_REVIEW\')\n                                            extracted=extract_source(body,crawl.get(\'content_type\') or \'\',\n                                                source_url=crawl.get(\'final_url\') or fetch,requested_url=fetch)\n                                            text=extracted[\'source_text\']\n                                            extra={\'snapshot_id\':crawl[\'snapshot_id\'],\'raw_sha256\':sha,\n                                                \'extract_status\':\'EXTRACT_SUCCESS\',\'text\':text,\'char_count\':len(text),\n                                                \'title\':extracted[\'title\'],\'language\':extracted[\'language\'],\n                                                \'language_method\':extracted[\'language_method\'],\'language_confidence\':extracted[\'language_confidence\'],\n                                                \'extractor\':extracted[\'parser\'],\'extractor_version\':POLICY,\n                                                \'source_issue\':None,\'text_markdown\':\'\',\'raw_hash_convention\':convention}\n                                            hold=_hold({**crawl,**extra,\'url\':fetch},2_000_000)\n                                            if hold: raise ValueError(\':\'.join(hold))\n                                            pending.append({\'fetch_url\':fetch,\'source_file_id\':remote[\'id\'],\n                                                \'leaf\':receipt[\'leaf\'][\'name\'],\'original_snapshot_id\':crawl[\'snapshot_id\'],\n                                                \'source_text_sha256\':hashlib.sha256(text.encode()).hexdigest(),\n                                                \'crawl_json\':raw,\'extracted_json\':json.dumps(extra,ensure_ascii=False)})\n                                            self._raw_result(task,\'EXTRACTED\')\n                                        except EXPECTED_PARSE_ERRORS as error:\n                                            self._raw_result(task,str(error)[:300])\n                            remaining_paths.discard(member.name)\n                        if headers%checkpoint_headers==0 or len(pending)>=32:\n                            save(offset+archive.offset)\n                        if time.monotonic()-logged>=15:\n                            print(\'RAW_RECOVERY_PROGRESS:\',receipt[\'leaf\'][\'name\'],\'|\',headers,\'headers |\',len(pending),\'new text pending save\',flush=True)\n                            logged=time.monotonic()\n                        archive.members.clear()\n                        if not remaining_paths:\n                            break\n                    for missing in remaining_paths:\n                        for task in by_path[missing]: self._raw_result(task,\'RAW_MEMBER_NOT_FOUND_IN_SOURCE\')\n                    save(offset+archive.offset,eof=True)\n            if identity(reader.metadata(remote[\'id\']))!=identity(remote):\n                raise ValueError(\'Raw source version changed during recovery; do not freeze.\')\n            self.commit(\'raw:\'+leaf_id,\'raw_source\',{\'leaf_id\':leaf_id,\'complete\':True})\n        self.commit(\'raw-pass\',\'raw_campaign\',{\'complete\':True,\'policy\':POLICY,\'updated_at\':utc_now()})\n        return self.status()\n\n    def _raw_result(self,task,result):\n        fetch,path,sha,raw,review=task\n        self.con.execute(\'INSERT OR REPLACE INTO raw_results VALUES (?,?,?,?)\',[sha,fetch,review,result])\n        if result!=\'EXTRACTED\':\n            self.con.execute("UPDATE todo SET reason=? WHERE fetch=? AND state=\'NEEDS_RAW\'",[result,fetch])\n\n    def export_remaining(self):\n        import csv\n        for name,query in (\n            (\'remaining_for_crawl.csv\',"SELECT a.id,a.url,t.state,t.reason FROM aliases a JOIN todo t USING(fetch) WHERE t.state IN (\'NEEDS_CRAWL\',\'NEEDS_RAW\',\'CHUNK_REVIEW\') ORDER BY a.id"),\n            (\'raw_recovery_outcomes.csv\',\'SELECT sha,source_url,quality_review,result FROM raw_results ORDER BY source_url,sha\')):\n            local=self.work/name\n            cursor=self.con.execute(query)\n            with local.open(\'w\',encoding=\'utf-8\',newline=\'\') as stream:\n                writer=csv.writer(stream); writer.writerow([c[0] for c in cursor.description])\n                for rows in iter(lambda:cursor.fetchmany(2048),[]): writer.writerows(rows)\n            publish_file(local,self.root/name)\n        return self.status()\n\n    def freeze_delta(self,phase,links,tokenizer,golden,*,audit_size=20):\n        key=\'freeze:\'+phase\n        if saved:=self.saved(key):\n            print(\'DELTA_FREEZE_REUSED:\',phase,\'| manifest only; no old corpus/inputs replay\',flush=True)\n            self._publish_delta_manifest()\n            return saved\n        if phase==\'raw-text-v1\' and not self.saved(\'raw-pass\'):\n            raise ValueError(\'Finish/resume raw recovery before freezing its delta.\')\n        origin="json_extract(candidate,\'$.recovery_root\')=\'maximize\'" if phase==\'raw-text-v1\' else "coalesce(json_extract(candidate,\'$.recovery_root\'),\'\')!=\'maximize\'"\n        if self.con.execute("SELECT count(*) FROM todo WHERE state=\'TEXT_READY\' AND "+origin).fetchone()[0]:\n            raise ValueError(\'Complete pending delta chunks before freezing.\')\n        target=self.root/\'processed\'/phase\n        if not (target/\'build.json\').is_file(): return None\n        build=read_json(target/\'build.json\')\n        if not build[\'counts\'].get(\'documents\'): return None\n        health=health_report(target,official_links=links,work_dir=self.work,audit_size=audit_size)\n        candidate=freeze_candidate(target,health,golden_report=golden)\n        name=\'candidate-\'+candidate[\'candidate_manifest_sha256\'][:16]+\'.json\'\n        inputs=prepare_index_inputs(target,name,tokenizer,work_dir=self.work)\n        saved={\'data_root\':str(self.root.relative_to(self.project)), \'build_run\':phase,\'candidate_name\':name,\n            \'candidate_manifest_sha256\':candidate[\'candidate_manifest_sha256\'],\n            \'snapshot_sha256\':candidate[\'snapshot_sha256\'],\'index_inputs_manifest_sha256\':inputs[\'manifest_sha256\'],\n            \'documents\':build[\'counts\'][\'documents\'],\'unique_dense_inputs\':inputs[\'input_count\'],\n            \'state\':\'DELTA_READY_FOR_EMBEDDING\',\'submission_candidate_changed\':False,\'updated_at\':utc_now()}\n        self.commit(key,\'freeze\',saved)\n        self._publish_delta_manifest()\n        return saved\n\n    def _publish_delta_manifest(self):\n        candidates=[json.loads(r[0]) for r in self.con.execute("SELECT receipt FROM completed WHERE kind=\'freeze\' ORDER BY id")]\n        atomic_json(self.root/\'delta_candidates.json\',seal({\'builds\':candidates,\'scope\':\'New recovery delta only\',\n            \'submission_candidate_changed\':False}))\n'
TOKENIZER_REVISION='5617a9f61b028005a4858fdac845db406aefb181'
CHUNK_POLICY={'child_tokens': 180, 'overlap_tokens': 40, 'parent_tokens': 512, 'snap_sentences': True, 'prefer_sections': True, 'dense_max_tokens': 512}

(WORK/'collect_all_btc_sources.py').write_text(COLLECT_SOURCE,encoding='utf-8')
(WORK/'maximize_collection.py').write_text(MAXIMIZE_SOURCE,encoding='utf-8')
import collect_all_btc_sources as collector
import maximize_collection as maximizer
collector=importlib.reload(collector)
maximizer=importlib.reload(maximizer)
import google.auth
from google.auth.transport.requests import AuthorizedSession
from vietmedbridge.chunks import ChunkConfig,load_bge_tokenizer
from vietmedbridge.golden import run_golden_suite
from vietmedbridge.artifacts import verify_file
credentials,_=google.auth.default()
reader=collector.CollectionDriveSource(AuthorizedSession(credentials))
COLLECTION=collector.CollectAll(PROJECT,WORK/'collection')
SPEC={'model_id':'BAAI/bge-m3','revision':TOKENIZER_REVISION,'special_tokens':False}
CONFIG=ChunkConfig(**CHUNK_POLICY)
FLOW=maximizer.MaximizeCollection(COLLECTION,WORK/'delta',tokenizer_spec=SPEC,chunk_config=CONFIG)
LAZY={}
def resources():
    if not LAZY:
        print('Load BGE tokenizer only; no GPU/model weights.',flush=True)
        tokenizer,spec=load_bge_tokenizer(TOKENIZER_REVISION)
        if spec!=SPEC: raise ValueError('Tokenizer policy differs from the paid corpus.')
        links=PROJECT/'data_staging/btc-new-71be0e12-20261019-v1/references/5586f423916005c1-links.parquet'
        verify_file(links,COLLECTION.binding['official_sha256'])
        golden=run_golden_suite(CHECKOUT/'tests/golden/cases.json',tokenizer,SPEC,CONFIG)
        if not golden['passed']: raise ValueError('Golden source-span regression failed.')
        LAZY.update(tokenizer=tokenizer,links=links,golden=golden)
    return LAZY
print(json.dumps(FLOW.status(),ensure_ascii=False,indent=2),flush=True)


In [ ]:
#@title 3. Ưu tiên text đã cứu: chunk/freeze/index-input chỉ phần mới
needs=FLOW.con.execute("SELECT count(*) FROM todo WHERE state='TEXT_READY' AND coalesce(json_extract(candidate,'$.recovery_root'),'')!='maximize'").fetchone()[0]
if needs or not FLOW.saved('freeze:metadata-text-v1'):
    r=resources()
    FLOW.chunk_ready(r['tokenizer'],phase='metadata-text-v1')
    RESULT=FLOW.freeze_delta('metadata-text-v1',r['links'],r['tokenizer'],r['golden'])
else:
    RESULT=FLOW.saved('freeze:metadata-text-v1')
print(json.dumps(RESULT,ensure_ascii=False,indent=2),flush=True)
FLOW.export_remaining()

In [ ]:
#@title 4. Tận dụng raw đã crawl: tiếp tục đúng cursor, giữ gate hash/challenge
try:
    FLOW.recover_raw(reader)
    RESULT=FLOW.saved('freeze:raw-text-v1')
    pending=FLOW.con.execute("SELECT count(*) FROM todo WHERE state='TEXT_READY' AND json_extract(candidate,'$.recovery_root')='maximize'").fetchone()[0]
    if not RESULT and (pending or (FLOW.root/'processed/raw-text-v1/build.json').is_file()):
        r=resources()
        FLOW.chunk_ready(r['tokenizer'],phase='raw-text-v1')
        RESULT=FLOW.freeze_delta('raw-text-v1',r['links'],r['tokenizer'],r['golden'])
    elif RESULT:
        FLOW._publish_delta_manifest()
    print(json.dumps(RESULT,ensure_ascii=False,indent=2),flush=True)
    print(json.dumps(FLOW.export_remaining(),ensure_ascii=False,indent=2),flush=True)
    print('Outputs:',FLOW.root,flush=True)
    print('delta_candidates.json: chỉ dữ liệu mới, cần GPU embedding rồi ghép vào retrieval.',flush=True)
    print('remaining_for_crawl.csv: URL chưa có nội dung đạt gate; gửi nhóm crawl theo nguyên nhân.',flush=True)
finally:
    FLOW.close()
    COLLECTION.close()